<a href="https://colab.research.google.com/github/JeremieTarantop/Cardiac-Diagnostic-CMR-report-/blob/main/notebooks/CMR_report_Colab_TestData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMR Report on Colab — No data upload needed

This notebook runs the **PTB-XL → CMR report** pipeline on Colab with GPU.

**You don’t need to download or upload any data.** It uses the small **test PTB data** that’s already in the repo (`data/ptbxl_test/`). The notebook converts that into the format the pipeline expects and runs it.

---

### Step 0: Open in Colab and turn on GPU

1. Click: **[Open in Colab](https://colab.research.google.com/github/JeremieTarantop/Cardiac-Diagnostic-CMR-report-/blob/main/notebooks/CMR_report_Colab_TestData.ipynb)**  
2. **Runtime → Change runtime type → T4 GPU** (or better) → Save.
3. Run the cells **in order** (Runtime → Run all, or run each cell with Shift+Enter).

## 1. Clone the repo

This brings in the code and the test data (`data/ptbxl_test/`).

In [1]:
!git clone https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-.git repo_cmr
%cd repo_cmr

Cloning into 'repo_cmr'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 188 (delta 14), reused 187 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 26.55 MiB | 16.53 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/repo_cmr


## 2. Install dependencies

Installs PyTorch, Transformers, and WFDB (to read the test ECG files).

In [2]:
!pip install -q transformers torch pandas numpy scipy wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 142.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.
cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.


## 3. Prepare test data (no upload needed)

Converts the WFDB files in `data/ptbxl_test/` into the format the pipeline expects and creates minimal metadata. Everything stays inside the repo.

In [3]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import wfdb
except ImportError:
    !pip install -q wfdb
    import wfdb

REPO = Path(".")
TEST_DIR = REPO / "data" / "ptbxl_test" / "00000"
OUT_NPY_DIR = REPO / "data" / "ptbxl_pclr_format" / "00000"
OUT_NPY_DIR.mkdir(parents=True, exist_ok=True)
(REPO / "data" / "ptbxl_with_labels").mkdir(parents=True, exist_ok=True)

TARGET_SAMPLES = 4096
N_LEADS = 12

hea_files = sorted(TEST_DIR.glob("*.hea"))
if not hea_files:
    raise FileNotFoundError(f"No .hea files in {TEST_DIR}. Is data/ptbxl_test in the repo?")

print(f"Found {len(hea_files)} test records in data/ptbxl_test/00000/")

labels_rows = []
db_rows = []

for i, hea_path in enumerate(hea_files[:50], start=1):  # up to 50
    rec_name = hea_path.stem
    rec_path = str(hea_path.with_suffix(""))
    rec = wfdb.rdrecord(rec_path)
    sig = rec.p_signal  # (n_samples, n_sig)
    if sig.shape[1] != N_LEADS:
        continue
    if sig.shape[0] >= TARGET_SAMPLES:
        sig = sig[:TARGET_SAMPLES, :]
    else:
        pad = np.zeros((TARGET_SAMPLES - sig.shape[0], N_LEADS), dtype=sig.dtype)
        sig = np.vstack([sig, pad])
    npy_path = OUT_NPY_DIR / f"{rec_name}.npy"
    np.save(npy_path, sig)
    rel_path = npy_path.relative_to(REPO)
    labels_rows.append({
        "ecg_id": i,
        "ecg_file": str(rel_path),
        "age": 50.0,
        "sex": 1,
        "height": "", "weight": "", "scp_codes": "{}", "heart_axis": "",
        "infarction_stadium1": "", "infarction_stadium2": "", "baseline_drift": "",
        "static_noise": "", "burst_noise": "", "electrodes_problems": "", "extra_beats": "", "pacemaker": "",
    })
    db_rows.append({
        "ecg_id": i,
        "patient_id": 0,
        "age": 50.0,
        "sex": 1,
        "height": "", "weight": "", "nurse": "", "site": "", "device": "", "recording_date": "",
        "report": f"Test record {i} from PTB-XL (WFDB)",
        "scp_codes": "{}",
        "heart_axis": "",
        "infarction_stadium1": "", "infarction_stadium2": "", "validated_by": "", "second_opinion": "",
        "initial_autogenerated_report": "", "validated_by_human": "",
        "baseline_drift": "", "static_noise": "", "burst_noise": "", "electrodes_problems": "", "extra_beats": "", "pacemaker": "",
        "strat_fold": "", "filename_lr": "", "filename_hr": "",
    })

pd.DataFrame(labels_rows).to_csv(REPO / "data" / "ptbxl_with_labels" / "ecg_with_labels.csv", index=False)
pd.DataFrame(db_rows).to_csv(REPO / "data" / "ptbxl_database.csv", index=False)

print(f"Created {len(labels_rows)} records in data/ptbxl_pclr_format and metadata CSVs.")
print("You can run the CMR pipeline with --ecg-id 1 to", len(labels_rows))
print("Done. Run the next cell.")

Found 50 test records in data/ptbxl_test/00000/
Created 50 records in data/ptbxl_pclr_format and metadata CSVs.
You can run the CMR pipeline with --ecg-id 1 to 50
Done. Run the next cell.


## 4. Generate the CMR report — TinyLlama (GPU)

Runs TinyLlama on the first test ECG and prints the CMR report. Uses GPU if available.

In [4]:
import os
os.environ["USE_TRANSFORMERS"] = "1"
os.environ["USE_CUDA"] = "1"

!python -m ecg_to_cmr_report.e_to_c_llama1 --ecg-id 1

<frozen runpy>:128: RuntimeWarning: 'ecg_to_cmr_report.e_to_c_llama1' found in sys.modules after import of package 'ecg_to_cmr_report', but prior to execution of 'ecg_to_cmr_report.e_to_c_llama1'; this may result in unpredictable behaviour
Loading PTB-XL record (metadata + ECG signal)...
ECG ID: 1  |  Report: Test record 1 from PTB-XL (WFDB)...
ECG signal: 12 leads loaded.

Calling local LLM (Transformers (CPU))...

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 (first time may download ~2GB)...
config.json: 100% 608/608 [00:00<00:00, 3.89MB/s]
tokenizer_config.json: 1.29kB [00:00, 5.97MB/s]
tokenizer.json: 1.84MB [00:00, 139MB/s]
tokenizer.model: 100% 500k/500k [00:01<00:00, 408kB/s]
special_tokens_map.json: 100% 551/551 [00:00<00:00, 3.95MB/s]
model.safetensors: 100% 2.20G/2.20G [00:03<00:00, 651MB/s]
Loading weights: 100% 201/201 [00:00<00:00, 1464.36it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 823kB/s]
Model loaded on cuda.
Generating 

## 4b. (Optional) Generate with MedGemma 4B (medical LLM, GPU)

Uses the same test data. **Requires a Hugging Face token:** accept [MedGemma terms](https://huggingface.co/google/medgemma-4b-it), then set your token in Colab (e.g. **Secrets** or run the line below with your token). Slower than TinyLlama but more medical wording.

Pass your token here··········
Logged in.


In [31]:
!pip -q install -U huggingface_hub


In [29]:
from huggingface_hub import login
import getpass

token = getpass.getpass("HF token (hidden): ")
login(token=token)
print("Logged in from Python.")


HF token (hidden): ··········
Logged in from Python.


In [30]:
from huggingface_hub import HfFolder, whoami
print("Token present:", bool(HfFolder.get_token()))
print("Account:", whoami())


ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

In [26]:
!huggingface-cli whoami

/bin/bash: line 1: huggingface-cli: command not found


In [20]:
!python - <<'PY'
from huggingface_hub import hf_hub_download
print("Trying to download config.json...")
p = hf_hub_download(repo_id="google/medgemma-4b-it", filename="config.json")
print("OK:", p)
PY


/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Trying to download config.json...


GatedRepoError: 401 Client Error. (Request ID: Root=1-69891a58-31fad22e65e1599e50d7d3da;78a371ee-ff36-42e3-8e7f-0dc6eaf3840b)

Cannot access gated repo for url https://huggingface.co/google/medgemma-4b-it/resolve/main/config.json.
Access to model google/medgemma-4b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [15]:
# Optional: set your HF token (or add it in Colab Secrets as HF_TOKEN)
# os.environ["HF_TOKEN"] = "your_hf_token_here"
!python -m ecg_to_cmr_report.e_to_c_medgemma4b --ecg-id 1 --max-tokens 128

Loading PTB-XL record (metadata + ECG signal)...
ECG ID: 1  |  Report: Test record 1 from PTB-XL (WFDB)...
ECG signal: 12 leads loaded.

Calling MedGemma 4B...

Loading google/medgemma-4b-it (first time may download ~8GB)...
If you get a 401, run: huggingface-cli login and accept the model terms at https://huggingface.co/google/medgemma-4b-it
Error: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
Time elapsed: 0 min 6 s


## 5. (Optional) Try another test record

Change `--ecg-id` to 2, 3, … up to the number of test records prepared above.

In [ ]:
# !python -m ecg_to_cmr_report.e_to_c_llama1 --ecg-id 2